## §1 — Esplorazione del dataset annotato

Carico il seed set (commenti annotati manualmente) e ispeziono:
- Distribuzione delle etichette STANCE (necessaria per calibrare i `class_weights`)
- Numero di etichettati vs non-etichettati (i secondi finiranno nell'inferenza massiva)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Il dataset contiene commenti annotati manualmente (seed set) + commenti non ancora etichettati
df = pd.read_csv(
    "/kaggle/input/datasets/francescochiarolanza/dataset-consenso-meloni/definitivo_governo.csv",
    encoding="utf-8-sig", sep=";", dtype=str, engine="python", on_bad_lines="skip"
)
df = df.drop(columns=[c for c in df.columns if "Unnamed" in c], errors="ignore")
print(f"Righe totali: {len(df):,}")
print(f"Colonne:      {df.columns.tolist()}")

# Normalizza la colonna STANCE (potrebbe avere case misto, NaN come stringa, ecc.)
df["STANCE"] = df["STANCE"].astype(str).str.strip().str.upper()
df["STANCE"] = df["STANCE"].replace({"NAN": np.nan, "NONE": np.nan, "": np.nan})

# Separa solo etichettati / non-etichettati. NON filtriamo per spam.
df_labeled   = df[df["STANCE"].notna()].copy()
df_unlabeled = df[df["STANCE"].isna()].copy()

print(f"\nEtichettati:     {len(df_labeled):>7,}  (usati per il training)")
print(f"Non etichettati: {len(df_unlabeled):>7,}  (passeranno all\'inferenza massiva)")

# Distribuzione STANCE
stance_counts = df_labeled["STANCE"].value_counts()
colors = {"POS": "#2ecc71", "NEG": "#e74c3c", "NEU": "#3498db"}

plt.figure(figsize=(8, 5))
plt.bar(stance_counts.index, stance_counts.values,
        color=[colors.get(s, "#999") for s in stance_counts.index])
plt.title("Distribuzione STANCE — Seed set annotato", fontsize=13, fontweight="bold")
plt.ylabel("Conteggio")
for i, (label, v) in enumerate(zip(stance_counts.index, stance_counts.values)):
    plt.text(i, v + 10, f"{v}\n({v/len(df_labeled)*100:.1f}%)", ha="center", fontsize=10)
plt.tight_layout()
plt.savefig("distribuzione_stance.png", dpi=150)
plt.show()

print("\n" + "="*50)
print("REPORT FINALE")
print("="*50)
for s in ["POS", "NEG", "NEU"]:
    n = (df_labeled["STANCE"] == s).sum()
    print(f"  {s}: {n:>5,}  ({n/len(df_labeled)*100:.1f}%)")

df_labeled.to_csv("seed_train.csv",  index=False, encoding="utf-8-sig", sep=";")
df_unlabeled.to_csv("unlabeled.csv", index=False, encoding="utf-8-sig", sep=";")
print(f"\nSalvato seed_train.csv ({len(df_labeled):,}) e unlabeled.csv ({len(df_unlabeled):,})")

## §2 — Ispezione qualitativa: campione 100 commenti NEG

Prima del training, lettura manuale di 100 commenti annotati come **NEG** per verificare la qualità dell'annotazione e calibrare il giudizio sul decision boundary NEG vs NEU (la coppia più difficile da discriminare).

Output: `campione_100_neg.csv` — scaricabile dal pannello `/working/` di Kaggle.

In [ ]:
import pandas as pd, os

FILE_PATH   = "/kaggle/input/datasets/francescochiarolanza/dataset-consenso-meloni/inferenza_definitiva.csv"
OUTPUT_PATH = "/kaggle/working/campione_100_neg.csv"

df = pd.read_csv(FILE_PATH, sep=";", dtype=str)
col_testo = "Testo" if "Testo" in df.columns else ("Commento" if "Commento" in df.columns else df.columns[1])

df_neg = df[df["STANCE"] == "NEG"]
n      = min(100, len(df_neg))
camp   = df_neg.sample(n=n, random_state=40)   # seed fisso per riproducibilità

camp[["Canale", col_testo, "STANCE"]].to_csv(OUTPUT_PATH, sep=";", index=False)
print(f"Salvato campione in {OUTPUT_PATH} ({n} commenti NEG)\n")

print("=" * 80)
print(f"CAMPIONE {n} COMMENTI NEG")
print("=" * 80)
for i, (_, row) in enumerate(camp.iterrows(), 1):
    print(f"[{i:>3}] Canale: {row.get('Canale','N/A')}")
    print(f"      Testo:  {str(row.get(col_testo,'')).replace(chr(10),' ')[:200]}")
    print("-" * 80)

## §3 — Fine-tuning BERT — Stance Detection Governo

Pipeline di training:

1. **Carica** il dataset annotato (con `Testo_Padre` per le risposte)
2. **Costruisci input** — `text_a` = padre (per le risposte) o testo (per i principali); `text_b` = risposta o vuoto. BERT inserisce automaticamente `[CLS] text_a [SEP] text_b [SEP]`.
3. **Encoding label** — `NEG=0, NEU=1, POS=2` (mappa diretta agli ID del modello sentiment pre-addestrato)
4. **Split stratificato 80/10/10** train/val/test
5. **Tokenizzazione** — `padding="max_length"`, `truncation=True`, `max_length=256` (lasciamo spazio per padre+risposta)
6. **Class weights** inversi alla frequenza per bilanciare classi rare
7. **Training** con `WeightedTrainer` (loss pesata) ed `EarlyStopping` su F1 macro
8. **Valutazione** su test set + confusion matrix
9. **Analisi per tipo** — accuracy commenti principali vs risposte (con/senza padre)
10. **Salvataggio** modello in `./model_stance_final/` + `label_info_stance.pkl`

### §3.0 — Installazione transformers

In [ ]:
# Fix ambiente Kaggle: il `peft` preinstallato attiva un import dentro
# `transformers.trainer` (`from peft import PeftModel`) che fallisce per
# disallineamenti interni a peft stesso post-upgrade. Questo notebook NON usa
# peft (nessun LoRA/adapter), quindi lo disinstalliamo e pinniamo transformers
# a una versione coerente. Senza peft, transformers gestisce il branch via
# `is_peft_available() == False`.
import sys
!pip uninstall -y -q peft
!pip install -q --upgrade "transformers==4.46.3" "accelerate>=0.34" "datasets>=2.20"

# Svuota la cache dei moduli già caricati (evita di dover riavviare il kernel)
for _mod in list(sys.modules):
    if _mod.startswith(("transformers", "peft", "accelerate")):
        del sys.modules[_mod]

import transformers, accelerate
print("transformers:", transformers.__version__)
print("accelerate: ", accelerate.__version__)
print("peft:         disinstallato (non serve per fine-tuning standard)")

### §3.1 — Training pipeline completo

In [ ]:
import pandas as pd
import numpy as np
import torch
import pickle
import shutil
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from datasets import Dataset
from torch import nn

# ============================================================
# CONFIGURAZIONE KAGGLE
# ============================================================
# Usa il modello base italiano
MODEL_NAME = "dbmdz/bert-base-italian-xxl-uncased"
# Sostituisci questo percorso con quello esatto del tuo dataset su Kaggle!
PATH_DATASET = "/kaggle/input/datasets/francescochiarolanza/dataset-consenso-meloni/definitivo_governo.csv"

MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS     = 10
TASK       = "stance_governo"
LABEL_COL  = "STANCE"

print(f"🎯 Task: {TASK}")
print(f"📦 Modello base: {MODEL_NAME}")

# ============================================================
# PULIZIA DISCO (Utile su Kaggle per non saturare lo spazio)
# ============================================================
for folder in [f"./model_{TASK}", f"./logs_{TASK}"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)

# ============================================================
# LABEL MAP
# ============================================================
LABEL_VALIDE = {"POS", "NEG", "NEU"}
LABEL_MAP = {"NEG": 0, "NEU": 1, "POS": 2}
LABEL_NAMES = ["NEG", "NEU", "POS"]

# ============================================================
# CARICAMENTO E PREPARAZIONE DATASET
# ============================================================
print("📂 Caricamento dataset...")
df = pd.read_csv(PATH_DATASET, encoding="utf-8-sig", sep=";", dtype=str, engine="python", on_bad_lines="skip")
df = df.drop(columns=[c for c in df.columns if "Unnamed" in c], errors="ignore")

df = df.dropna(subset=["Testo", LABEL_COL])
df["Testo"] = df["Testo"].astype(str).str.strip()
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.upper()

has_padre = "Testo_Padre" in df.columns
has_titolo = "Titolo_Video" in df.columns

df["text_a"] = ""
df["text_b"] = ""

for i, row in df.iterrows():
    commento = str(row["Testo"]).strip()
    titolo = str(row.get("Titolo_Video", "")).strip() if has_titolo else ""
    padre = ""

    if has_padre and str(row.get("Tipo_Commento", "")) == "Risposta" and pd.notna(row.get("Testo_Padre")):
        padre = str(row["Testo_Padre"]).strip()

    if titolo and padre and len(padre) > 3:
        contesto = titolo[:80] + " | " + padre[:200]
    elif titolo:
        contesto = titolo[:150]
    elif padre and len(padre) > 3:
        contesto = padre[:250]
    else:
        contesto = ""

    df.at[i, "text_a"] = contesto if contesto else commento
    df.at[i, "text_b"] = commento if contesto else ""

df = df[df["text_a"].str.len() > 3]
df = df[df[LABEL_COL].isin(LABEL_VALIDE)].copy()
df["label"] = df[LABEL_COL]
df["label_id"] = df["label"].map(LABEL_MAP)

print(f"\n📊 Campioni disponibili per l'addestramento: {len(df)}")

# ============================================================
# SPLIT STRATIFICATO 80/10/10
# ============================================================
df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label_id"])
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42, stratify=df_temp["label_id"])
print(f"🔀 Split — Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

# ============================================================
# TOKENIZZAZIONE (Con fix 'only_first')
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    texts_a = batch["text_a"]
    texts_b = batch["text_b"]
    texts_b_clean = [b if b and len(b) > 0 else None for b in texts_b]

    final_a, final_b = [], []
    for a, b in zip(texts_a, texts_b_clean):
        final_a.append(a)
        final_b.append(b if b else "")

    return tokenizer(
        final_a,
        final_b,
        padding="max_length",
        truncation=True,  # 🔥 FIX CRITICO: Taglia solo il contesto, mai il commento!
        max_length=MAX_LENGTH
    )

def to_dataset(df_split):
    ds = Dataset.from_dict({
        "text_a": df_split["text_a"].tolist(),
        "text_b": df_split["text_b"].fillna("").tolist(),
        "labels": df_split["label_id"].tolist()
    })
    return ds.map(tokenize, batched=True)

ds_train = to_dataset(df_train)
ds_val   = to_dataset(df_val)
ds_test  = to_dataset(df_test)

# ============================================================
# INIZIALIZZAZIONE MODELLO E PESI
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n💻 Device attivo: {device}")

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True)
model.to(device)

counts = df_train["label_id"].value_counts().sort_index().values
class_weights = torch.tensor(1.0 / counts, dtype=torch.float)
class_weights = class_weights / class_weights.sum() * 3
# Non spostiamo i pesi sul device qui, lo facciamo dinamicamente nel Trainer!

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":    accuracy_score(labels, preds),
        "f1_macro":    f1_score(labels, preds, average="macro")
    }

# ============================================================
# CUSTOM TRAINER (Con fix GPU dinamica)
# ============================================================
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        
        # 🔥 FIX CRITICO: Allinea dinamicamente i pesi al device dei logit
        weights_on_device = class_weights.to(logits.device)
        
        loss    = nn.CrossEntropyLoss(weight=weights_on_device)(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir                  = f"./model_{TASK}",
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    save_total_limit            = 1,
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_macro",
    greater_is_better           = True,
    logging_dir                 = f"./logs_{TASK}",
    logging_steps               = 50,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    learning_rate               = 1e-5,
    fp16                        = torch.cuda.is_available(),
    report_to                   = "none",
)

trainer = WeightedTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = ds_train,
    eval_dataset    = ds_val,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)]
)

# ============================================================
# AVVIO ADDESTRAMENTO E VALUTAZIONE
# ============================================================
print(f"\n🚀 Avvio addestramento...")
trainer.train()

print(f"\n📊 Valutazione finale sul Test Set Cieco:")
predictions = trainer.predict(ds_test)
preds       = np.argmax(predictions.predictions, axis=-1)
labels_test = predictions.label_ids

print(classification_report(labels_test, preds, target_names=LABEL_NAMES, digits=3, zero_division=0))

# Salvataggio Matrice di Confusione
cm = confusion_matrix(labels_test, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, cmap="Blues")
plt.title(f"Confusion Matrix — {TASK}")
plt.ylabel("Reale")
plt.xlabel("Predetto")
plt.tight_layout()
plt.savefig(f"/kaggle/working/cm_{TASK}.png", dpi=150)
print(f"📉 Matrice di confusione salvata come cm_{TASK}.png")

# ============================================================
# SALVATAGGIO MODELLO FINALE
# ============================================================
print("\n💾 Salvataggio del modello vincitore...")
trainer.save_model(f"/kaggle/working/model_{TASK}_final")
tokenizer.save_pretrained(f"/kaggle/working/model_{TASK}_final")

print("✅ Operazione conclusa! Il modello è salvato e pronto per l'inferenza.")

In [ ]:
# ============================================================
# COMPRESSIONE E DOWNLOAD (SICUREZZA KAGGLE)
# ============================================================
import shutil
from IPython.display import FileLink, display

cartella_modello = f"/kaggle/working/model_{TASK}_final"
nome_zip         = f"modello_{TASK}_definitivo"

print("\n⏳ Compressione del modello in corso... (potrebbe volerci circa un minuto)")
shutil.make_archive(nome_zip, "zip", cartella_modello)

print(f"✅ Compressione completata! Clicca il link qui sotto per scaricarlo:")
display(FileLink(f"{nome_zip}.zip"))

### §3.2 — Compressione e download del modello (opzionale)

Comprime il modello salvato in `.zip` per il download da Kaggle (necessario per esportarlo come dataset/model riutilizzabile).

In [ ]:
import shutil
from IPython.display import FileLink

cartella_modello = "/kaggle/working/model_stance_final"
nome_zip         = "/kaggle/working/modello_stance_xxl"

print("⏳ Compressione del modello in corso... (BERT XXL pesa ~1.3GB)")
shutil.make_archive(nome_zip, "zip", cartella_modello)

print("✅ Compressione completata.")
FileLink(r"modello_stance_xxl.zip")

Il codice sottostante mostra le metriche del modello senza la necessità di rifare il training 

In [ ]:
  import os
  os.environ["WANDB_DISABLED"] = "true"
  os.environ["WANDB_MODE"]     = "disabled"

  import pandas as pd, numpy as np, json
  import seaborn as sns, matplotlib.pyplot as plt
  from sklearn.model_selection import train_test_split
  from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
  from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                            Trainer, TrainingArguments)
  from datasets import Dataset

  MODEL_DIR  = "/kaggle/input/models/francescochiarolanza/modello-stance-referendum/other/default/2/modello_stance_xxl V2"
  DATA_PATH  = "/kaggle/input/datasets/francescochiarolanza/commenti-youtube-referendum-giustizia/Dataset Tesi Aggiornato.csv"
  MAX_LENGTH = 256
  LABEL_MAP   = {"NEG": 0, "NEU": 1, "POS": 2}
  LABEL_NAMES = ["NEG", "NEU", "POS"]

  # --- Ricarica dataset con stesso preprocessing del training ---
  df = pd.read_csv(DATA_PATH, encoding="utf-8-sig", sep=";", dtype=str,
                   engine="python", on_bad_lines="skip")
  df = df.drop(columns=[c for c in df.columns if "Unnamed" in c], errors="ignore")
  df = df.dropna(subset=["Testo", "STANCE"])
  df["Testo"]  = df["Testo"].astype(str).str.strip()
  df["STANCE"] = df["STANCE"].astype(str).str.strip().str.upper()

  df["text_a"] = np.nan
  df["text_b"] = np.nan
  if "Testo_Padre" in df.columns:
      for i, row in df.iterrows():
          padre = str(row.get("Testo_Padre") or "").strip()
          if row.get("Tipo_Commento") == "Risposta" and len(padre) > 3:
              df.at[i, "text_a"] = padre
              df.at[i, "text_b"] = row["Testo"]
          else:
              df.at[i, "text_a"] = row["Testo"]
  else:
      df["text_a"] = df["Testo"]

  df = df[df["text_a"].astype(str).str.len() > 3]
  df = df[df["STANCE"].isin(LABEL_MAP)].copy()
  df["label_id"] = df["STANCE"].map(LABEL_MAP)

  # --- Stesso split 80/10/10 con seed 42 → estrae il test set ---
  df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label_id"])
  df_val,   df_test = train_test_split(df_temp, test_size=0.5, random_state=42, stratify=df_temp["label_id"])
  print(f"Test set ricostruito: {len(df_test)} esempi")
  print("Distribuzione classi (test):")
  print(df_test["STANCE"].value_counts())

  # --- Carica modello + tokenizer ---
  tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
  model     = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

  def tokenize(batch):
      tb = [b if b is not None else "" for b in batch["text_b"]]
      return tokenizer(batch["text_a"], tb,
                       padding="max_length", truncation=True, max_length=MAX_LENGTH)

  ds_test = Dataset.from_dict({
      "text_a": df_test["text_a"].tolist(),
      "text_b": [b if pd.notna(b) else None for b in df_test["text_b"]],
      "labels": df_test["label_id"].tolist(),
  }).map(tokenize, batched=True)

  # --- Trainer con report_to="none" → niente wandb / tensorboard / hub ---
  eval_args = TrainingArguments(
      output_dir="./_eval_tmp",
      per_device_eval_batch_size=32,
      report_to="none",
      logging_strategy="no",
      save_strategy="no",
      do_train=False,
      do_eval=False,
  )
  trainer = Trainer(model=model, args=eval_args, tokenizer=tokenizer)

  # --- Inferenza ---
  out    = trainer.predict(ds_test)
  y_pred = np.argmax(out.predictions, axis=-1)
  y_true = np.array(df_test["label_id"])

  # --- Metriche ---
  print("\n" + "="*60)
  print("CLASSIFICATION REPORT")
  print("="*60)
  print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=3))
  print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
  print(f"F1 macro : {f1_score(y_true, y_pred, average='macro'):.4f}")
  print(f"F1 weight: {f1_score(y_true, y_pred, average='weighted'):.4f}")

  # --- Confusion matrix ---
  cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
  fig, axes = plt.subplots(1, 2, figsize=(13, 5))

  sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
              xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
              cbar=False, ax=axes[0])
  axes[0].set_xlabel("Predetto"); axes[0].set_ylabel("Reale")
  axes[0].set_title("Confusion Matrix — conteggi assoluti")

  cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
  sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Greens",
              xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
              cbar=False, ax=axes[1])
  axes[1].set_xlabel("Predetto"); axes[1].set_ylabel("Reale")
  axes[1].set_title("Confusion Matrix — recall per classe (riga = 100%)")

  plt.tight_layout()
  plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150, bbox_inches="tight")
  plt.show()

  # --- Storico training (se presente trainer_state.json) ---
  ts_path = os.path.join(MODEL_DIR, "trainer_state.json")
  if os.path.exists(ts_path):
      with open(ts_path) as f:
          state = json.load(f)
      print("\n" + "="*60)
      print("STORICO TRAINING (per epoca)")
      print("="*60)
      for log in state.get("log_history", []):
          if "eval_f1_macro" in log:
              print(f"  epoch {log['epoch']:>4.1f} → "
                    f"F1 macro {log['eval_f1_macro']:.4f} | "
                    f"acc {log.get('eval_accuracy', float('nan')):.4f} | "
                    f"loss {log.get('eval_loss', float('nan')):.4f}")
  else:
      print(f"\n(Nessun trainer_state.json in {MODEL_DIR})")

## §4 — Inferenza Massiva sul dataset YouTube Referendum

Una volta che il modello è stato (eventualmente) raffinato con active learning, lo applichiamo a **tutti** i commenti del dataset (etichettati e non) per produrre la colonna `PREDIZIONE_STANCE` usata in tutte le analisi del notebook 3.

**Output**: `risultati_inferenza_completa.csv` con tutte le colonne originali + `PREDIZIONE_STANCE` (POS/NEG/NEU) e `CONFIDENZA` (probabilità della classe predetta).

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import os

# ==========================================
# CONFIGURAZIONE
# ==========================================
WORKING_DIR    = "/kaggle/working/"
FULL_DATA_FILE = "/kaggle/input/datasets/francescochiarolanza/commenti-youtube-referendum-giustizia/Dataset Tesi.csv"
MODEL_DIR      = "/kaggle/input/models/francescochiarolanza/modello-stance-referendum/other/default/2/modello_stance_xxl V2"
OUTPUT_CSV     = os.path.join(WORKING_DIR, "risultati_inferenza_completa.csv")

ID2LABEL   = {0: "NEG", 1: "NEU", 2: "POS"}
MAX_LENGTH = 256
BATCH_SIZE = 32     # se va in OOM abbassa a 16

print("🚀 Inizio pipeline di inferenza massiva...")

# ==========================================
# 1. CARICAMENTO DATI
# ==========================================
df = pd.read_csv(FULL_DATA_FILE, sep=";", dtype=str, encoding="utf-8-sig")

# Prepara coppie (padre, risposta) coerenti con il training
df["text_a"] = df["Testo"].fillna("").astype(str).str.strip()
df["text_b"] = None

if "Testo_Padre" in df.columns and "Tipo_Commento" in df.columns:
    for i, row in df.iterrows():
        if row.get("Tipo_Commento") == "Risposta" and pd.notna(row.get("Testo_Padre")):
            padre = str(row["Testo_Padre"]).strip()
            if len(padre) > 3:
                df.at[i, "text_a"] = padre
                df.at[i, "text_b"] = row["Testo"]

# Filtro commenti troppo corti (< 4 caratteri = inutili anche con BERT)
df = df[df["Testo"].astype(str).str.len() > 3].copy().reset_index(drop=True)
print(f"📊 Commenti da analizzare: {len(df):,}")

# ==========================================
# 2. MODELLO
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚙️ Caricamento modello da: {MODEL_DIR} su {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()


# ==========================================
# 3. INFERENZA BATCH
# ==========================================
class InferenceDataset(Dataset):
    """Dataset PyTorch per inferenza con coppie (text_a, text_b)."""
    def __init__(self, df, tokenizer, max_length):
        self.texts_a   = df["text_a"].tolist()
        self.texts_b   = df["text_b"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts_a)

    def __getitem__(self, idx):
        tb = self.texts_b[idx]
        encoding = self.tokenizer(
            self.texts_a[idx],
            tb if tb is not None else "",
            truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt",
        )
        return {k: v.squeeze(0) for k, v in encoding.items()}


dataset    = InferenceDataset(df, tokenizer, MAX_LENGTH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

all_preds, all_probs = [], []
with torch.no_grad():
    for batch in tqdm(dataloader, desc="Inferenza"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        probs   = F.softmax(outputs.logits, dim=-1)
        max_p, preds = torch.max(probs, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(max_p.cpu().numpy())

# ==========================================
# 4. ASSEMBLA RISULTATI
# ==========================================
df["PREDIZIONE_STANCE"] = [ID2LABEL[p] for p in all_preds]
df["CONFIDENZA"]        = all_probs
df = df.drop(columns=["text_a", "text_b"], errors="ignore")

df.to_csv(OUTPUT_CSV, sep=";", index=False, encoding="utf-8-sig")
print(f"\n✅ Inferenza completata. Output: {OUTPUT_CSV}")
print(f"\nDistribuzione PREDIZIONE_STANCE:")
print(df["PREDIZIONE_STANCE"].value_counts())
print(f"\nConfidenza media: {df['CONFIDENZA'].mean():.3f}")
print(f"Confidenza < 0.6:  {(df['CONFIDENZA'] < 0.6).sum():,} ({(df['CONFIDENZA'] < 0.6).mean()*100:.1f}%)")

## §5 — Fine-tuning BERT — Stance Detection Referendum


**Input** stesso del referendum: `[CLS] contesto [SEP] commento [SEP]` dove `contesto = titolo[:80] | padre[:200]`.


In [ ]:
import pandas as pd
import numpy as np
import torch
import pickle, shutil, os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
from datasets import Dataset
from torch import nn

# ============================================================
# CONFIGURAZIONE — Domain adaptation
# ============================================================
# Parte dal modello referendum se esiste, altrimenti dal BERT italiano base
#MODEL_NAME = "/kaggle/working/model_stance_final"
#if not os.path.exists(MODEL_NAME):
MODEL_NAME = "dbmdz/bert-base-italian-xxl-uncased"
print(f"Modello referendum non trovato, uso base: {MODEL_NAME}")
#else:
    #print(f"Domain adaptation da: {MODEL_NAME}")

MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS     = 10
TASK       = "stance_referendum"
LABEL_COL  = "STANCE"
LABEL_VALIDE = {"POS", "NEG", "NEU"}
LABEL_MAP    = {"NEG": 0, "NEU": 1, "POS": 2}
LABEL_NAMES  = ["NEG", "NEU", "POS"]

# Pulizia checkpoint precedenti
for folder in [f"./model_{TASK}", f"./logs_{TASK}"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)

# ============================================================
# CARICAMENTO
# ============================================================
df = pd.read_csv(
    "/kaggle/input/datasets/francescochiarolanza/dataset-consenso-meloni/definitivo_governo.csv",
    encoding="utf-8-sig", dtype=str, engine="python", on_bad_lines="skip",
)
if len(df.columns) <= 3:
    df = pd.read_csv(
        "/kaggle/input/datasets/francescochiarolanza/dataset-consenso-meloni/definitivo_governo.csv",
        encoding="utf-8-sig", sep=";", dtype=str, engine="python", on_bad_lines="skip",
    )
df = df.drop(columns=[c for c in df.columns if "Unnamed" in c], errors="ignore")
print(f"Colonne: {df.columns.tolist()}")
print(f"Righe: {len(df)}")

# ============================================================
# COSTRUZIONE INPUT — text_a = contesto, text_b = commento
# ============================================================
# text_a riceve titolo[:80] + padre[:200] (o solo titolo, o solo padre, o niente)
# text_b riceve il commento da classificare. Se text_a è vuoto, mettiamo il
# commento in text_a e text_b vuoto (single-sentence input).
# truncation="longest_first" tronca il segmento più lungo (di solito il contesto),
# preservando sempre il commento.
df = df.dropna(subset=["Testo", LABEL_COL])
df["Testo"]   = df["Testo"].astype(str).str.strip()
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.upper()

has_padre  = "Testo_Padre" in df.columns
has_titolo = "Titolo_Video" in df.columns
df["text_a"] = ""
df["text_b"] = ""

for i, row in df.iterrows():
    commento = str(row["Testo"]).strip()
    titolo   = str(row.get("Titolo_Video", "")).strip() if has_titolo else ""
    padre    = ""
    if has_padre and str(row.get("Tipo_Commento", "")) == "Risposta" and pd.notna(row.get("Testo_Padre")):
        padre = str(row["Testo_Padre"]).strip()

    if titolo and padre and len(padre) > 3:
        contesto = titolo[:80] + " | " + padre[:200]
    elif titolo:
        contesto = titolo[:150]
    elif padre and len(padre) > 3:
        contesto = padre[:250]
    else:
        contesto = ""

    df.at[i, "text_a"] = contesto if contesto else commento
    df.at[i, "text_b"] = commento if contesto else ""

df = df[df["text_a"].str.len() > 3]
df = df[df[LABEL_COL].isin(LABEL_VALIDE)].copy()
df["label"]    = df[LABEL_COL]
df["label_id"] = df["label"].map(LABEL_MAP)

n_ctx = (df["text_b"].str.len() > 0).sum()
print(f"\nCon contesto (titolo/padre): {n_ctx:,}")
print(f"Senza contesto:               {len(df)-n_ctx:,}")
print(df["label"].value_counts())

# ============================================================
# SPLIT 80/10/10
# ============================================================
df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label_id"])
df_val,   df_test = train_test_split(df_temp, test_size=0.5, random_state=42, stratify=df_temp["label_id"])
print(f"Split — Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

# ============================================================
# TOKENIZZAZIONE
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text_a"],
        [b if b and len(b) > 0 else None for b in batch["text_b"]],
        padding="max_length",
        truncation="longest_first",   # tronca il contesto, non il commento
        max_length=MAX_LENGTH,
    )

def to_dataset(df_split):
    ds = Dataset.from_dict({
        "text_a": df_split["text_a"].tolist(),
        "text_b": df_split["text_b"].fillna("").tolist(),
        "labels": df_split["label_id"].tolist(),
    })
    return ds.map(tokenize, batched=True)

ds_train = to_dataset(df_train)
ds_val   = to_dataset(df_val)
ds_test  = to_dataset(df_test)

# ============================================================
# MODELLO + CLASS WEIGHTS
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")

# ignore_mismatched_sizes=True permette di caricare un modello con head a 3 classi
# anche se in qualche checkpoint i nomi dei layer differiscono leggermente
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True,
)
model.to(device)

counts = df_train["label_id"].value_counts().sort_index().values
class_weights = torch.tensor(1.0 / counts, dtype=torch.float)
class_weights = class_weights / class_weights.sum() * 3
print("Class weights (più alto = classe più rara):")
for name, w in zip(LABEL_NAMES, class_weights.tolist()):
    print(f"  {name}: {w:.3f}")

# ============================================================
# TRAINING
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":    accuracy_score(labels, preds),
        "f1_macro":    f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        # I pesi devono essere sul device corrente (per multi-GPU)
        loss = nn.CrossEntropyLoss(weight=class_weights.to(outputs.logits.device))(
            outputs.logits, labels
        )
        return (loss, outputs) if return_outputs else loss


training_args = TrainingArguments(
    output_dir                  = f"./model_{TASK}",
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_macro",
    greater_is_better           = True,
    logging_dir                 = f"./logs_{TASK}",
    logging_steps               = 50,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    learning_rate               = 1e-5,    # BASSO: domain adaptation, non training da zero
    fp16                        = torch.cuda.is_available(),
    report_to                   = "none",
)

trainer = WeightedTrainer(
    model=model, args=training_args,
    train_dataset=ds_train, eval_dataset=ds_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"\n🚀 Avvio training {TASK}...")
trainer.train()

# ============================================================
# VALUTAZIONE
# ============================================================
print(f"\n📊 Valutazione finale (test set cieco):")
pred_out    = trainer.predict(ds_test)
preds       = np.argmax(pred_out.predictions, axis=-1)
labels_test = pred_out.label_ids

print(classification_report(labels_test, preds, target_names=LABEL_NAMES, digits=3, zero_division=0))

cm = confusion_matrix(labels_test, preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, cmap="Blues")
plt.title(f"Confusion Matrix — {TASK}")
plt.ylabel("Reale"); plt.xlabel("Predetto")
plt.tight_layout()
plt.savefig(f"cm_{TASK}.png", dpi=150)
plt.show()

# Accuracy con/senza contesto
df_eval = df_test.copy()
df_eval["pred"]    = [LABEL_NAMES[p] for p in preds]
df_eval["correct"] = df_eval["label"] == df_eval["pred"]
has_ctx = df_eval["text_b"].str.len() > 0
print(f"\nAccuracy con contesto:   {df_eval[has_ctx]['correct'].mean():.3f}  ({has_ctx.sum()} campioni)")
print(f"Accuracy senza contesto: {df_eval[~has_ctx]['correct'].mean():.3f}  ({(~has_ctx).sum()} campioni)")

# ============================================================
# SALVATAGGIO
# ============================================================
trainer.save_model(f"./model_{TASK}_final")
tokenizer.save_pretrained(f"./model_{TASK}_final")

with open(f"label_info_{TASK}.pkl", "wb") as f:
    pickle.dump({
        "task": TASK, "model": MODEL_NAME,
        "label_names": LABEL_NAMES, "label_map": LABEL_MAP,
        "num_labels": 3, "max_length": MAX_LENGTH,
        "uses_title_context": has_titolo, "uses_parent_context": has_padre,
    }, f)

f1 = f1_score(labels_test, preds, average="macro")
print(f"\n{'='*50}")
print(f"RIEPILOGO {TASK}")
print(f"{'='*50}")
print(f"F1 macro:  {f1:.3f}")
print(f"Accuracy:  {accuracy_score(labels_test, preds):.3f}")
print(f"Modello salvato: ./model_{TASK}_final")

## §6 — Ottimizzazione Soglia ed Error Analysis

Dopo il training, due analisi:

1. **Error analysis**: matrice confusion + campione errori a bassa confidenza per identificare pattern sistematici.
2. **Threshold tuning sulla classe NEU**: NEU è la classe più difficile da discriminare. Predico NEU solo se `P(NEU) ≥ thresh`, altrimenti scelgo tra NEG/POS. Provo soglie 0.3 → 0.7 per trovare il punto che massimizza F1 macro.

Questo aiuta a ridurre i falsi positivi su NEU senza penalizzare le altre classi.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score

MODEL_PATH = "/kaggle/working/model_stance_governo_final"
DATA_PATH  = "/kaggle/input/datasets/francescochiarolanza/dataset-consenso-meloni/definitivo_governo.csv"

# ── Ricostruisce ESATTAMENTE lo stesso test set del training ──────────────────
# Stesso seed=42 → stessi campioni (riproducibilità)
df = pd.read_csv(DATA_PATH, sep=";", dtype=str, engine="python", on_bad_lines="skip")
df = df.dropna(subset=["Testo", "STANCE"])

has_padre  = "Testo_Padre" in df.columns
has_titolo = "Titolo_Video" in df.columns
df["text_a"] = ""
df["text_b"] = ""

for i, row in df.iterrows():
    commento = str(row["Testo"]).strip()
    titolo   = str(row.get("Titolo_Video", "")).strip() if has_titolo else ""
    padre    = ""
    if has_padre and str(row.get("Tipo_Commento", "")) == "Risposta" and pd.notna(row.get("Testo_Padre")):
        padre = str(row["Testo_Padre"]).strip()
    if titolo and padre and len(padre) > 3:
        contesto = titolo[:80] + " | " + padre[:200]
    elif titolo:
        contesto = titolo[:150]
    elif padre and len(padre) > 3:
        contesto = padre[:250]
    else:
        contesto = ""
    df.at[i, "text_a"] = contesto if contesto else commento
    df.at[i, "text_b"] = commento if contesto else ""

df = df[df["text_a"].str.len() > 3]
LABEL_MAP   = {"NEG": 0, "NEU": 1, "POS": 2}
LABEL_NAMES = ["NEG", "NEU", "POS"]
df["label_id"] = df["STANCE"].map(LABEL_MAP)

df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label_id"])
_, df_test = train_test_split(df_temp, test_size=0.5, random_state=42, stratify=df_temp["label_id"])
print(f"Test set: {len(df_test)} campioni")

# ── Inferenza con probabilità ─────────────────────────────────────────────────
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()

all_probs, all_labels = [], []
batch_size = 32

for i in range(0, len(df_test), batch_size):
    batch = df_test.iloc[i:i+batch_size]
    enc   = tokenizer(
        batch["text_a"].tolist(),
        [b if b and len(b) > 0 else None for b in batch["text_b"].tolist()],
        padding=True, truncation="longest_first", max_length=256, return_tensors="pt",
    ).to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    all_probs.extend(F.softmax(logits, dim=-1).cpu().numpy())
    all_labels.extend(batch["label_id"].tolist())

probs  = np.array(all_probs)
labels = np.array(all_labels)
preds  = np.argmax(probs, axis=-1)

print("\nReport con soglia standard (argmax):")
print(classification_report(labels, preds, target_names=LABEL_NAMES, digits=3, zero_division=0))

# ── Error analysis ────────────────────────────────────────────────────────────
df_eval = df_test.copy().reset_index(drop=True)
df_eval["pred"]    = [LABEL_NAMES[p] for p in preds]
df_eval["label"]   = df_eval["STANCE"]
df_eval["correct"] = df_eval["label"] == df_eval["pred"]
df_eval["conf"]    = probs.max(axis=1)

print("\n--- ERRORI PIÙ FREQUENTI (label reale vs predizione) ---")
errori = df_eval[~df_eval["correct"]]
print(errori.groupby(["label", "pred"]).size().sort_values(ascending=False).head(10))

print("\n--- CAMPIONE ERRORI A BASSA CONFIDENZA (conf < 0.6) ---")
bassa_conf = errori[errori["conf"] < 0.6].head(10)
for _, r in bassa_conf.iterrows():
    print(f"  Reale: {r['label']} | Pred: {r['pred']} | Conf: {r['conf']:.2f}")
    print(f"  {str(r['text_a'])[:120]}")
    print()

# ── Ottimizzazione soglia per NEU ─────────────────────────────────────────────
# Predizione "tweaked": NEU solo se P(NEU) >= thresh, altrimenti argmax(NEG, POS)
print("\n--- OTTIMIZZAZIONE SOGLIA NEU ---")
print(f"{'Soglia':>8} {'F1 macro':>10} {'Accuracy':>10}")
for thresh in [0.3, 0.4, 0.5, 0.6, 0.7]:
    adjusted = []
    for prob in probs:
        if prob[1] >= thresh:                              # P(NEU) supera la soglia → NEU
            adjusted.append(1)
        else:
            adjusted.append(np.argmax([prob[0], 0, prob[2]]))   # tra NEG (0) e POS (2)
    f1  = f1_score(labels, adjusted, average="macro")
    acc = accuracy_score(labels, adjusted)
    print(f"  {thresh:>8.1f} {f1:>10.3f} {acc:>10.3f}")

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"]     = "disabled"

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, 
                          Trainer, TrainingArguments)
from datasets import Dataset

# ============================================================
# 1. CONFIGURAZIONI E PERCORSI
# ============================================================
# Usa il path esatto della cartella non zippata che contiene i pesi
MODEL_DIR  = "/kaggle/input/models/francescochiarolanza/consenso-meloni/other/default/4/modello_stance_governo_definitivo"
DATA_PATH  = "/kaggle/input/datasets/francescochiarolanza/dataset-consenso-meloni/definitivo_governo.csv"

MAX_LENGTH   = 256
LABEL_VALIDE = {"POS", "NEG", "NEU"}
LABEL_MAP    = {"NEG": 0, "NEU": 1, "POS": 2}
LABEL_NAMES  = ["NEG", "NEU", "POS"]

# ============================================================
# 2. CARICAMENTO MODELLO E TOKENIZER
# ============================================================
print(f"🧠 Caricamento tokenizer e modello salvato da: {MODEL_DIR}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

# ============================================================
# 3. RICARICA DATASET E PREPROCESSING
# ============================================================
print("📂 Caricamento e preprocessing dataset...")
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig", sep=";", dtype=str, engine="python", on_bad_lines="skip")
df = df.drop(columns=[c for c in df.columns if "Unnamed" in c], errors="ignore")

df = df.dropna(subset=["Testo", "STANCE"])
df["Testo"]  = df["Testo"].astype(str).str.strip()
df["STANCE"] = df["STANCE"].astype(str).str.strip().str.upper()

has_padre  = "Testo_Padre" in df.columns
has_titolo = "Titolo_Video" in df.columns

df["text_a"] = ""
df["text_b"] = ""

for i, row in df.iterrows():
    commento = str(row["Testo"]).strip()
    titolo   = str(row.get("Titolo_Video", "")).strip() if has_titolo else ""
    padre    = ""

    if has_padre and str(row.get("Tipo_Commento", "")) == "Risposta" and pd.notna(row.get("Testo_Padre")):
        padre = str(row["Testo_Padre"]).strip()

    if titolo and padre and len(padre) > 3:
        contesto = titolo[:80] + " | " + padre[:200]
    elif titolo:
        contesto = titolo[:150]
    elif padre and len(padre) > 3:
        contesto = padre[:250]
    else:
        contesto = ""

    df.at[i, "text_a"] = contesto if contesto else commento
    df.at[i, "text_b"] = commento if contesto else ""

df = df[df["text_a"].str.len() > 3]
df = df[df["STANCE"].isin(LABEL_VALIDE)].copy()
df["label_id"] = df["STANCE"].map(LABEL_MAP)

# ============================================================
# 4. SPLIT 80/10/10 — IDENTICO AL TRAINING
# ============================================================
df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label_id"])
df_val,   df_test = train_test_split(df_temp, test_size=0.5, random_state=42, stratify=df_temp["label_id"])

print(f"✅ Test set ricostruito con successo: {len(df_test)} esempi")

# ============================================================
# 5. TOKENIZZAZIONE E PREPARAZIONE INFERENZA
# ============================================================
def tokenize(batch):
    texts_a = batch["text_a"]
    texts_b = [b if b and len(b) > 0 else None for b in batch["text_b"]]

    final_a, final_b = [], []
    for a, b in zip(texts_a, texts_b):
        final_a.append(a)
        final_b.append(b if b else "")

    return tokenizer(
        final_a,
        final_b,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

def to_dataset(df_split):
    ds = Dataset.from_dict({
        "text_a": df_split["text_a"].tolist(),
        "text_b": df_split["text_b"].fillna("").tolist(),
        "labels": df_split["label_id"].tolist()
    })
    return ds.map(tokenize, batched=True)

print("\n🔄 Tokenizzazione del test set...")
ds_test = to_dataset(df_test)

# ============================================================
# 6. INFERENZA (FIX APPLICATO QUI)
# ============================================================
eval_args = TrainingArguments(
    output_dir="./_eval_tmp",
    per_device_eval_batch_size=32,
    report_to="none",
    logging_strategy="no",
    save_strategy="no",
    do_train=False,
    do_eval=False,
)

# 🔥 IL FIX DEFINITIVO È IN QUESTA RIGA:
trainer = Trainer(model=model, args=eval_args, processing_class=tokenizer)

print("\n🚀 Avvio inferenza sul test set...")
out    = trainer.predict(ds_test)
y_pred = np.argmax(out.predictions, axis=-1)
y_true = out.label_ids

# ============================================================
# 7. OUTPUT METRICHE E GRAFICI
# ============================================================
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=3))
print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
print(f"F1 macro : {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"F1 weight: {f1_score(y_true, y_pred, average='weighted'):.4f}")

# Creazione subplot
cm     = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            cbar=False, ax=axes[0])
axes[0].set_xlabel("Predetto")
axes[0].set_ylabel("Reale")
axes[0].set_title("Confusion Matrix — Conteggi assoluti")

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Greens",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            cbar=False, ax=axes[1])
axes[1].set_xlabel("Predetto")
axes[1].set_ylabel("Reale")
axes[1].set_title("Confusion Matrix — Recall (riga = 100%)")

plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix_eval.png", dpi=150, bbox_inches="tight")
print("\n📉 Immagine salvata come 'confusion_matrix_eval.png'")

In [ ]:
# Dopo trainer.train(), forza il salvataggio del best model con verifica
best_metric = trainer.state.best_metric
best_checkpoint = trainer.state.best_model_checkpoint
print(f"\n🏆 Best checkpoint: {best_checkpoint}")
print(f"🏆 Best F1 macro:   {best_metric}")

# Salva esplicitamente anche i pesi con il mapping corretto
model.config.id2label = {0: "NEG", 1: "NEU", 2: "POS"}
model.config.label2id = {"NEG": 0, "NEU": 1, "POS": 2}
model.config.num_labels = 3
trainer.save_model("/kaggle/working/model_stance_governo_final")
tokenizer.save_pretrained("/kaggle/working/model_stance_governo_final")
print("✅ Modello salvato con label mapping corretto")